# AIFS-driven TIFNet: CPU inference


This example runs the trained model for tropical cyclone **202424 (Man-yi)**, initialized at **2024-11-14 12:00 UTC** in the paper's Man-yi case-study figure. It begins with centered, interpolated, **unnormalized** arrays, not raw global weather files. Future CMA observations are for comparison only.

## 1. Environment

Before running this notebook, run `python -m pip install -e .` from the repository root in your Python environment. Required packages: NumPy, SciPy, ONNX Runtime, pandas, Matplotlib and Jupyter. PyTorch is not required. Launch from the repository root or its `notebooks/` directory.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import onnxruntime as ort
from tifnet.inference import download_verified_model, load_sample_inputs

repository_root = Path.cwd().resolve()
if repository_root.name == "notebooks":
    repository_root = repository_root.parent
if not (repository_root / "models/manifest.json").is_file():
    raise FileNotFoundError("Launch from the repository root or its notebooks directory.")
input_source = "aifs"  # Change to "ifs" and rerun all cells to switch BOTH model and sample.
sample_directory = repository_root / "data/202424_2024111412"
metadata = json.loads((sample_directory / "metadata.json").read_text())
manifest = json.loads((repository_root / "models/manifest.json").read_text())

## 2. Download and verify the model

The fixed v1.0.0 URL, file size and SHA256 are recorded in the manifest. A valid local file is reused. An existing file with a wrong checksum raises an error; it is never silently used or overwritten.

In [ ]:
model_filepath = download_verified_model(manifest[input_source], repository_root / "models")
print(f"Verified: {model_filepath.name}")

## 3. Load the processed sample and inspect its contract

Read `README.md` for physical units and preprocessing. Channel order and shapes are checked below. The ONNX graph includes normalization and maximum-wind reconstruction; do not normalize the inputs again.

In [ ]:
input_arrays = load_sample_inputs(sample_directory / f"{input_source}_inputs.npz", metadata)
print("Surface variables:", metadata["surface_variables"])
print("Surface units:", metadata["surface_units"])
print("Isobaric variables:", metadata["isobaric_variables"])
print("Isobaric units:", metadata["isobaric_units"])
print("Pressure levels (hPa):", metadata["pressure_levels_hpa"])
print("History offsets (h):", metadata["historical_time_offsets_hours"])
pd.DataFrame([{"input": name, "shape": values.shape, "dtype": str(values.dtype)}
              for name, values in input_arrays.items()])

## 4. Run CPU inference

The CPU setting below flushes subnormal floating-point values to zero to avoid very slow arithmetic on some processors.

In [ ]:
session_options = ort.SessionOptions()
session_options.intra_op_num_threads = 4
# Avoid slow CPU arithmetic on subnormal float32 values.
session_options.add_session_config_entry("session.set_denormal_as_zero", "1")
session = ort.InferenceSession(str(model_filepath), sess_options=session_options,
                               providers=["CPUExecutionProvider"])
for specification in session.get_inputs():
    if specification.name not in input_arrays or list(input_arrays[specification.name].shape) != specification.shape:
        raise ValueError(f"Model/sample interface mismatch: {specification.name}")
predicted_maximum_wind_speed = session.run(None, input_arrays)[0]
if predicted_maximum_wind_speed.shape != (1, 21) or not np.isfinite(predicted_maximum_wind_speed).all():
    raise ValueError("Expected 21 finite predictions in m/s.")

## 5. Forecast table and paper reference

The table includes the original paper predictions and CMA observations alongside the ONNX forecast. Reference values are copied from the original paper evaluation CSV.

In [ ]:
forecast_table = pd.read_csv(sample_directory / "reference_predictions.csv")
if forecast_table.forecast_lead_time_hours.tolist() != list(range(0, 121, 6)):
    raise ValueError("Reference lead times must be 0–120 h at 6 h intervals.")
reference_prediction = forecast_table[f"{input_source}_paper_maximum_wind_speed_ms"].to_numpy()
forecast_table["onnx_maximum_wind_speed_ms"] = predicted_maximum_wind_speed[0]
output_directory = repository_root / "outputs"
output_directory.mkdir(exist_ok=True)
forecast_table.to_csv(output_directory / f"{input_source}_forecast.csv", index=False)
forecast_table

## 6. Intensity evolution

This single-case curve compares the TIFNet prediction with CMA observations; it is not an aggregate skill estimate. All 21 lead times are shown. Only a PDF is saved.

In [ ]:
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "pdf.fonttype": 42,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False})
figure, axis = plt.subplots(figsize=(7.2, 4.2), layout="constrained")
forecast_lead_time = forecast_table.forecast_lead_time_hours
axis.plot(forecast_lead_time, forecast_table.observed_maximum_wind_speed_ms,
          color="0.25", marker="o", markersize=3, label="CMA observations")
axis.plot(forecast_lead_time, predicted_maximum_wind_speed[0],
          color="#3575A5", marker="s", markersize=3, label=f"{input_source.upper()}-driven TIFNet")
axis.set(xlabel="Forecast lead time (h)", ylabel="Maximum wind speed (m/s)",
         title="Man-yi (202424) | Initialized 2024-11-14 12:00 UTC", xlim=(0, 120))
axis.set_xticks(range(0, 121, 24))
axis.legend()
figure.savefig(output_directory / f"{input_source}_intensity.pdf")
plt.show()